# FaceForge — Conditional DDPM Training (Kaggle, T4 x2)

Trains the attribute-conditioned DDPM on CelebA at **128x128**. This notebook **imports the
project's own modules** (`model.py`, `dataset.py`, `utils.py`, `train.py`) from the GitHub repo
rather than redefining them, so the code that trains here is exactly the code in the repo.

> **Budget warning.** 128x128 is ~4x the pixels of 64x64 and forces a much smaller batch. On T4 x2
> expect roughly **1-3 hours per epoch over the full 202K images** — so 50 epochs does not fit in
> Kaggle's ~30h weekly quota. **Time your first epoch and extrapolate before committing.** If it's
> too slow, set `samples_per_epoch` (training cell) so each epoch trains on a fresh random subset
> instead of the full 202K — this bounds per-epoch time while the model still sees the whole
> dataset given enough epochs, unlike permanently capping to one fixed slice of images. Or train at
> 64x64 first and treat 128 as a later step. Diffusion models need many passes; too few epochs at
> high resolution looks worse than more epochs at low resolution.

### Setup before running
1. **Add data**: Add Input → search `jessicali9530/celeba-dataset` → Add.
2. **Accelerator**: Settings → Accelerator → **GPU T4 x2**.
3. **Internet**: Settings → Internet → **On** (needed to `git clone` the repo).
4. *(Resuming only)* Add your previous run's output checkpoint as an input dataset and set `RESUME_CHECKPOINT`.

### Conditioning attributes
The model conditions on these **10 CelebA binary attributes**, embedded by `UNet.label_mlp` and summed
with the timestep embedding so every conv block sees the conditioning signal:

| Category | Attributes |
|---|---|
| Gender | `Male` |
| Age | `Young` |
| Expression | `Smiling` |
| Hair | `Black_Hair`, `Blond_Hair`, `Brown_Hair`, `Bald` |
| Facial hair | `Mustache`, `Goatee` |
| Accessories | `Eyeglasses` |

Each is 0/1 (CelebA's `-1/1` is remapped in `dataset.load_attributes`). The canonical order is
`dataset.SELECTED_ATTRIBUTES` — attribute vectors must always follow it.

### Architecture notes
The U-Net uses pre-activation residual blocks with SiLU, GroupNorm, strided-conv downsampling and
upsample+conv (no transposed conv, avoiding checkerboard artifacts). Blocks at 128 channels and
above use a squeeze-3x3-expand (ResNet-v2 bottleneck) design instead of two full-width 3x3 convs —
the wide convs otherwise dominate the parameter count for little benefit, since the 1x1 projections
already carry the full channel width. Self-attention runs only at 16x16 and 8x8 — applying it at
every resolution is ~5x slower for no real benefit. At 128x128 the five levels run at
128 -> 64 -> 32 -> 16 -> 8, so attention still lands on the same two. The output conv is
zero-initialised and has **no** activation, since the training target is noise ~ N(0,1).

CelebA images are 178x218, so they are centre-cropped to a square **before** resizing — resizing
straight to a square squashes every face vertically by ~22%. Horizontal flips are enabled as free
augmentation (all 10 conditioning attributes are flip-invariant).

Training uses a **cosine** noise schedule, an **EMA** of the weights for sampling, and
**classifier-free guidance**: `cond_drop_prob` of samples train against a learned null token, so at
sampling time `guidance_scale > 1` extrapolates away from the unconditional prediction and sharpens
attribute adherence. CFG costs 2 forward passes per denoising step when sampling.

### Notes
- Do **not** `pip install torch`/`torchvision`; Kaggle's preinstalled CUDA build is already correct.
- Both GPUs are used via `nn.DataParallel`, handled inside `train.py`.
- Sessions cap at ~9-12h with a ~30h/week GPU quota, so `train.py` checkpoints every epoch and can resume.

In [1]:
!nvidia-smi

Tue Aug  4 13:20:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.65.06              Driver Version: 580.65.06      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               On  |   00000000:18:00.0 Off |                  Off |
| 35%   69C    P2            155W /  300W |    6389MiB /  49140MiB |     97%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Clone the project code

In [2]:
# import os, shutil

# REPO_URL = 'https://github.com/cjaitej/Face-Generation-using-Diffusion-Model.git'
# REPO_DIR = '/kaggle/working/FaceForge'

# # Re-clone so the notebook always trains the latest committed code.
# # NOTE: this wipes REPO_DIR, including any sample grids from earlier in *this* session.
# # Harmless at session start; avoid re-running this cell after training.
# if os.path.exists(REPO_DIR):
#     shutil.rmtree(REPO_DIR)
# !git clone -q $REPO_URL $REPO_DIR

# os.chdir(REPO_DIR)
# print('working dir:', os.getcwd())
# !ls

## 2. Locate the CelebA mount

The mount path depends on how the dataset was attached, so resolve it rather than hardcoding.
This deliberately never lists the 200k-image folder itself — only cheap `isdir` probes — because
listing it on Kaggle's network-backed filesystem is slow.

In [3]:
# import os

# def candidate_roots(base='/kaggle/input', max_children=50):
#     roots = []
#     if os.path.isdir(base):
#         for name in sorted(os.listdir(base))[:max_children]:
#             path = os.path.join(base, name)
#             roots.append(path)
#             try:  # handle an owner/slug nesting level, e.g. /kaggle/input/datasets/<owner>/<slug>
#                 if os.path.isdir(path):
#                     for sub in sorted(os.listdir(path))[:max_children]:
#                         sub_path = os.path.join(path, sub)
#                         roots.append(sub_path)
#                         if os.path.isdir(sub_path):
#                             for leaf in sorted(os.listdir(sub_path))[:max_children]:
#                                 roots.append(os.path.join(sub_path, leaf))
#             except (PermissionError, OSError):
#                 pass
#     return roots


# def resolve_celeba():
#     for root in candidate_roots():
#         attr = os.path.join(root, 'list_attr_celeba.csv')
#         if not os.path.exists(attr):
#             continue
#         # Images sit either one level deep (img_align_celeba/img_align_celeba) or flat.
#         nested = os.path.join(root, 'img_align_celeba', 'img_align_celeba')
#         flat = os.path.join(root, 'img_align_celeba')
#         for img_dir in (nested, flat):
#             if os.path.isdir(img_dir):
#                 return img_dir, attr
#     return None, None


# IMG_DIR, ATTR_FILE = resolve_celeba()
# if IMG_DIR is None:
#     print('Could not locate CelebA. Contents of /kaggle/input:')
#     for name in os.listdir('/kaggle/input'):
#         print('  ', name)
#     raise SystemExit('Add the "celeba-dataset" dataset via Add Input, then re-run this cell.')

IMG_DIR = "./img_align_celeba/img_align_celeba"
ATTR_FILE = "./list_attr_celeba.csv"

print('IMG_DIR  :', IMG_DIR)
print('ATTR_FILE:', ATTR_FILE)

IMG_DIR  : ./img_align_celeba/img_align_celeba
ATTR_FILE: ./list_attr_celeba.csv


## 3. Build the image list

`list_attr_celeba.csv` already enumerates every image in the dataset (that's one row per filename),
so the list comes straight from the CSV we're loading anyway -- no need to scan the image directory
at all, which would be slow on Kaggle's network-backed filesystem for no reason.

In [4]:
import os
from dataset import list_celeba_filenames

filenames = list_celeba_filenames(ATTR_FILE)
with open('data_set.txt', 'w') as f:
    f.writelines(os.path.join(IMG_DIR, name) + '\n' for name in filenames)

n = len(filenames)
print('images listed:', n)

images listed: 202599


## 4. Sanity-check the data pipeline

In [5]:
from dataset import SELECTED_ATTRIBUTES
from utils import get_data

class Args:
    pass

args = Args()
args.image_size = 128
args.center_crop = 178
args.random_flip = False
args.num_workers = 0
args.pin_memory = False
args.dataset_path = 'data_set.txt'
args.attr_file = ATTR_FILE
args.batch_size = 8

_dl = get_data(args)
_images, _attrs = next(iter(_dl))
print('images :', _images.shape, _images.dtype, f'range [{_images.min():.2f}, {_images.max():.2f}]')
print('attrs  :', _attrs.shape, _attrs.dtype)
print('order  :', SELECTED_ATTRIBUTES)
print('sample :', [int(v) for v in _attrs[0].tolist()])

images : torch.Size([8, 3, 128, 128]) torch.float32 range [-1.00, 1.00]
attrs  : torch.Size([8, 10]) torch.float32
order  : ['Male', 'Young', 'Smiling', 'Black_Hair', 'Blond_Hair', 'Brown_Hair', 'Bald', 'Mustache', 'Goatee', 'Eyeglasses']
sample : [0, 1, 1, 1, 0, 0, 0, 0, 0, 0]


## 5. Train

`train.py` handles DataParallel across both T4s, mixed precision, per-epoch checkpointing, and
periodic sample grids. Checkpoints go to `/kaggle/working` so they survive as run output.

`samples_per_epoch` caps each epoch to a fresh random subset of that many images (redrawn every
epoch, no repeats within an epoch) instead of iterating the full 202K -- this bounds per-epoch time
on Kaggle's quota while still covering the whole dataset given enough epochs, unlike permanently
restricting training to one fixed slice. Set it to `None` to use the full list every epoch.

Every `sample_every` epochs it saves a grid to `results/<run_name>/epoch_XXXX.jpg`. Each grid uses
`n_eval_samples` **freshly randomized** attribute combos (`dataset.random_attribute_batch`) and a
noise seed derived from the epoch number, so you see the model's general range instead of the same
handful of faces every time — re-running training reproduces the same sequence of previews, but
different epochs show different combos. Sampling runs the full 1000-step reverse loop, so keep
`sample_every` at 5+.

Lower `batch_size` if you hit OOM. `epochs` is the *target* total — resuming continues from the
checkpoint's epoch, so you can raise it across sessions.

In [ ]:
from train import train
import torch

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

args = Args()
args.run_name = 'FaceForge_Conditional'
args.epochs = 501
args.batch_size = 512 * max(torch.cuda.device_count(), 1)   # raise if VRAM allows; lower on OOM
args.image_size = 128
args.center_crop = 178              # CelebA is 178x218 — crop square before resizing
args.random_flip = True
NUM_WORKERS = min(4, os.cpu_count() or 2)  # Kaggle typically provides 4 CPU cores
args.num_workers = NUM_WORKERS
args.pin_memory = True
args.dataset_path = 'data_set.txt'
args.samples_per_epoch = 60000      # fresh random subset each epoch; None = full list every epoch
args.attr_file = ATTR_FILE          # set to None to train the unconditional baseline
args.device = 'cuda' if torch.cuda.is_available() else 'cpu'
args.lr = 1e-4
args.use_amp = torch.cuda.is_available()
args.time_emb_dim = 256
args.dropout = 0.0
args.schedule = 'cosine'            # or 'linear' for the original schedule
args.min_snr_gamma = 5.0             # Min-SNR-gamma loss weighting clip (Hang et al., 2023)
args.use_ema = True                 # sample from EMA weights (cleaner results)
args.ema_decay = 0.999
args.cond_drop_prob = 0.1           # classifier-free guidance conditioning dropout
args.guidance_scale = 3.0           # guidance strength for the preview grids
args.sample_every = 10              # save a preview grid every N epochs
args.sample_seed = 1234             # base seed for preview randomization (offset by epoch)
args.n_eval_samples = 8              # number of randomized faces per preview grid

# Set to a checkpoint path (e.g. from a previous session's output added as an input dataset) to resume.
RESUME_CHECKPOINT = './models/faceforge_checkpoint.pth.tar'
args.resume_checkpoint = RESUME_CHECKPOINT
args.checkpoint_path = './models/faceforge_checkpoint.pth.tar'

print('GPUs:', torch.cuda.device_count(), '| batch size:', args.batch_size)
train(args)

GPUs: 1 | batch size: 512

Loaded checkpoint from epoch 98.

Model parameters: 21,260,035


epoch 98/500: 100%|██████████| 117/117 [01:35<00:00,  1.23it/s, avg=0.0090, loss=0.0093]


epoch 98 avg loss: 0.0090


epoch 99/500: 100%|██████████| 117/117 [01:33<00:00,  1.25it/s, avg=0.0090, loss=0.0086]


epoch 99 avg loss: 0.0090


epoch 100/500: 100%|██████████| 117/117 [01:33<00:00,  1.25it/s, avg=0.0091, loss=0.0091]


epoch 100 avg loss: 0.0091


Saved samples to results/FaceForge_Conditional/epoch_0100.jpg


epoch 101/500:  11%|█         | 13/117 [00:11<01:22,  1.26it/s, avg=0.0090, loss=0.0090]

## 6. Epochs vs results — training progression

In [ ]:
import glob
import matplotlib.pyplot as plt

grids = sorted(glob.glob(os.path.join('results', args.run_name, 'epoch_*.jpg')))
print(f'{len(grids)} sample grids')

for path in grids:
    plt.figure(figsize=(18, 2.6))
    plt.imshow(plt.imread(path))
    plt.axis('off')
    plt.title(os.path.basename(path))
    plt.show()

## 7. Generate faces from chosen attributes

`build_attribute_vector` comes from `dataset.py` (same helper `predict.py` uses). Pass any subset of
`SELECTED_ATTRIBUTES`; omitted ones default to 0, and unknown names raise rather than silently
doing nothing. Set `seed` to an int to reproduce an identical batch.

In [ ]:
from model import Diffusion
from dataset import build_attribute_vector
from utils import save_images

ckpt = torch.load(args.checkpoint_path, map_location=args.device, weights_only=False)
model = (ckpt.get('ema_model') or ckpt['model']).to(args.device)   # EMA weights sample cleaner
print('checkpoint epoch:', ckpt['epoch'])

REQUESTED = {'Male': 1, 'Smiling': 1, 'Eyeglasses': 1}
n = 8
SEED = None             # set an int for reproducible output
GUIDANCE_SCALE = 3.0    # 1.0 = no guidance; higher = stronger attribute adherence

diffusion = Diffusion(img_size=args.image_size, device=args.device)
attrs = build_attribute_vector(REQUESTED, n).to(args.device)
samples = diffusion.sample(model, n, attributes=attrs, seed=SEED, guidance_scale=GUIDANCE_SCALE)
save_images(samples, '/kaggle/working/generated.jpg')

plt.figure(figsize=(18, 2.6))
plt.imshow(plt.imread('/kaggle/working/generated.jpg'))
plt.axis('off')
plt.title(str(REQUESTED))
plt.show()